# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

for col in ["impressions_90d", "ctr", "engagement_rate", "avg_position"]:
    s = df[col]
    print(f"{col:16s} min={s.min():>8.2f}  p25={s.quantile(.25):>8.2f}  median={s.median():>8.2f}  "
          f"p75={s.quantile(.75):>8.2f}  p99={s.quantile(.99):>10.2f}  max={s.max():>10.2f}  skew={s.skew():>6.2f}")

print()
print(f"engagement_rate == 0 exactly: {(df.engagement_rate==0).mean():.1%} of all rows")
print(f"ctr == 0 exactly:             {(df.ctr==0).mean():.1%} of all rows")

# The flyrank-data gotcha: avg_position == 0 means "not measured", not "rank 0".
zero_pos = df[df.avg_position == 0]
print(f"\navg_position == 0: {len(zero_pos)} rows ({len(zero_pos)/len(df):.1%} of all rows)")
print("their impressions_90d:")
print(zero_pos["impressions_90d"].describe()[["mean", "50%", "max"]])

impressions_90d  min=    1.00  p25=   81.00  median=  731.00  p75= 3615.25  p99=  73505.83  max= 517715.00  skew= 11.38
ctr              min=    0.00  p25=    0.00  median=    0.07  p75=    0.29  p99=      8.33  max=    100.00  skew= 17.44
engagement_rate  min=    0.00  p25=    0.00  median=    0.00  p75=    1.35  p99=     33.33  max=    100.00  skew=  7.22
avg_position     min=    0.00  p25=    6.20  median=   10.80  p75=   22.30  p99=     69.90  max=    245.00  skew=  1.98

engagement_rate == 0 exactly: 72.1% of all rows
ctr == 0 exactly:             44.0% of all rows

avg_position == 0: 1205 rows (4.0% of all rows)
their impressions_90d:
mean     1.884647
50%      1.000000
max     36.000000
Name: impressions_90d, dtype: float64


**Reading the numbers:** `impressions_90d` (skew 11.4) and `ctr` (skew 17.4) are both severely heavy-tailed — a few giant pages, a long tail of near-zero ones — exactly the flyrank-data warning. Both `ctr` and `engagement_rate` are also **zero-inflated**: roughly half the rows sit at exactly 0.00, not near zero. Any median or correlation on these fields has to account for that mass at zero, or it silently misbehaves (see Section 3). And `avg_position == 0` rows have essentially no real impressions (median 1, max 36) — confirming the flyrank-data note that 0 here means *not measured*, not *ranked #0*. My ML-04/ML-07 pools already exclude these rows correctly.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1 — "Better-ranked pages get higher CTR."** The whole basis for comparing CTR *within* a position tier (as my ML-07 baseline does) is that position tier is the dominant driver of CTR. If that's not even roughly true, tier-relative scoring doesn't mean much.

**Test 2 — "High-impression-volume pages have lower CTR."** My ML-07 baseline's top-20 review found the ranked queue dominated by high-volume pages, and I assumed that was because big pages tend to underperform on CTR. This test checks whether that assumption is actually true in the data.

**Test 3 — "Missingness in `avg_position` tracks `content_type`."** The flyrank-data skill warns that missingness isn't random — testing it before any fillna decision.

In [2]:
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
tier_order = ["top_3", "page_1", "striking"]

print("=== Test 1: CTR by position tier ===")
g = pool.groupby("position_tier")["ctr"].agg(["median", "mean", "count"]).reindex(tier_order)
print(g)
print("VERDICT: MIXED — striking (median 0.17) is clearly worse than page_1/top_3, which broadly "
      "supports tier-relative comparison. But top_3 (0.20) does NOT beat page_1 (0.24) as the simple "
      "story predicts — even with n=458, comfortably above the 50-row floor. Position tier still "
      "matters, but 'better rank -> strictly higher CTR' is not clean enough to lean on outside the "
      "tier buckets I already use.")

print("\n=== Test 2: impressions vs ctr (log1p impressions, Spearman) ===")
pool["log_impr"] = np.log1p(pool.impressions_90d)
rho, p = stats.spearmanr(pool["log_impr"], pool["ctr"])
print(f"pool-wide: rho={rho:.3f}  p={p:.2e}  n={len(pool)}")
for tier in tier_order:
    sub = pool[pool.position_tier == tier]
    r, p2 = stats.spearmanr(sub["log_impr"], sub["ctr"])
    print(f"  within {tier:9s}: rho={r:.3f}  p={p2:.2e}  n={len(sub)}")
print("VERDICT: OPPOSITE — the correlation is positive everywhere I sliced it (0.11 to 0.45), not "
      "negative. High-volume pages skew toward HIGHER CTR on average, not lower. My ML-07 weak-pick "
      "note last time was half-right for the wrong reason: high-volume pages dominate the ranked queue "
      "purely because score = gap x impressions rewards volume directly, not because volume itself "
      "predicts a worse CTR.")

print("\n=== Test 3: missingness in avg_position by content_type ===")
share = df.groupby("content_type").apply(lambda d: (d.avg_position == 0).mean())
counts = df.groupby("content_type").size()
for ct in share.index:
    print(f"  {ct:20s} missing_rate={share[ct]:.3f}  n={counts[ct]}")
print("VERDICT: CONFIRMED — feedly article is missing avg_position 34.8% of the time vs. 1.7% for "
      "keyword article and 0% for comparison article, all comfortably above the sample floor. A blind "
      "fillna(0) on avg_position would quietly encode 'is this a feedly article' as a feature.")

=== Test 1: CTR by position tier ===
               median      mean  count
position_tier                         
top_3            0.20  0.346572    458
page_1           0.24  0.338808   7064
striking         0.17  0.266798   4485
VERDICT: MIXED — striking (median 0.17) is clearly worse than page_1/top_3, which broadly supports tier-relative comparison. But top_3 (0.20) does NOT beat page_1 (0.24) as the simple story predicts — even with n=458, comfortably above the 50-row floor. Position tier still matters, but 'better rank -> strictly higher CTR' is not clean enough to lean on outside the tier buckets I already use.

=== Test 2: impressions vs ctr (log1p impressions, Spearman) ===
pool-wide: rho=0.213  p=4.29e-123  n=12023
  within top_3    : rho=0.452  p=1.78e-24  n=458
  within page_1   : rho=0.113  p=1.80e-21  n=7064
  within striking : rho=0.249  p=2.26e-64  n=4485
VERDICT: OPPOSITE — the correlation is positive everywhere I sliced it (0.11 to 0.45), not negative. High-volume pa

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

My own ML-07 baseline has a flag worth stress-testing: `weak_engagement_confirms`, which assumes `engagement_rate` falling below its tier median is an *independent* signal from CTR — evidence pointing the same direction from a metric the score itself doesn't use. If `engagement_rate` is basically just CTR with extra steps, that flag is decoration, not corroboration.

In [3]:
print("=== Is engagement_rate independent of ctr? ===")
for tier in tier_order:
    sub = pool[pool.position_tier == tier]
    rho, p = stats.spearmanr(sub["ctr"], sub["engagement_rate"])
    print(f"  {tier:9s}: spearman(ctr, engagement_rate)={rho:.3f}  p={p:.2e}  n={len(sub)}")
rho, p = stats.spearmanr(pool["ctr"], pool["engagement_rate"])
print(f"  pool-wide: rho={rho:.3f}  p={p:.2e}  n={len(pool)}")
print("-> MIXED: rho 0.27-0.49 is a real, positive correlation, not full redundancy (rho would be "
      "near 1) but not independent either. Calling it an 'independent' check overstates it.")

print("\n=== Why did last time's precision@K look so weak (base rate 0.5%)? ===")
tier_med_raw = pool.groupby("position_tier")["engagement_rate"].median()
print("raw tier-median engagement_rate:", tier_med_raw.round(3).to_dict())
for tier, med in tier_med_raw.items():
    sub = pool[pool.position_tier == tier]
    zero_share = (sub.engagement_rate == 0).mean()
    window = ((sub.engagement_rate > 0) & (sub.engagement_rate < med)).mean()
    print(f"  {tier:9s}: median={med:.3f}  zero_share={zero_share:.3f}  "
          f"share landing in the open window (0, median)={window:.4f}")
print("VERDICT: FALSE, as implemented. Over half of top_3 and striking rows have engagement_rate "
      "exactly 0, which drags their tier median itself down to 0.0 -- making the window "
      "'0 < engagement_rate < tier_median' mathematically EMPTY for 2 of 3 tiers. The flag could "
      "only ever fire on page_1 rows, and barely (1.4% of them). This is the zero-inflation trap "
      "from Section 1 catching my own baseline.")

print("\n=== The fix: median conditional on engagement_rate > 0 (the 'measured' rows) ===")
cond_med = pool[pool.engagement_rate > 0].groupby("position_tier")["engagement_rate"].median()
print("conditional median:", cond_med.round(3).to_dict())

pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]
queue = pool[pool.ctr < pool.tier_median_ctr].sort_values("lost_clicks_90d", ascending=False).reset_index(drop=True)

queue["confirmed_weak_old"] = (queue.engagement_rate > 0) & (queue.engagement_rate < queue.position_tier.map(tier_med_raw))
queue["confirmed_weak_fixed"] = (queue.engagement_rate > 0) & (queue.engagement_rate < queue.position_tier.map(cond_med))

old_base = queue["confirmed_weak_old"].mean()
fixed_base = queue["confirmed_weak_fixed"].mean()
print(f"\nbase rate: old={old_base:.4f}  fixed={fixed_base:.4f}")
for k in [20, 50, 100, 200]:
    old_p = queue["confirmed_weak_old"].head(k).mean()
    fixed_p = queue["confirmed_weak_fixed"].head(k).mean()
    print(f"  precision@{k:<4} old={old_p:.3f}  fixed={fixed_p:.3f}  "
          f"(fixed lift x{fixed_p/fixed_base:.1f} over fixed base rate)")

=== Is engagement_rate independent of ctr? ===
  top_3    : spearman(ctr, engagement_rate)=0.487  p=1.19e-28  n=458
  page_1   : spearman(ctr, engagement_rate)=0.274  p=1.32e-121  n=7064
  striking : spearman(ctr, engagement_rate)=0.336  p=1.16e-118  n=4485
  pool-wide: rho=0.315  p=3.05e-275  n=12023
-> MIXED: rho 0.27-0.49 is a real, positive correlation, not full redundancy (rho would be near 1) but not independent either. Calling it an 'independent' check overstates it.

=== Why did last time's precision@K look so weak (base rate 0.5%)? ===
raw tier-median engagement_rate: {'page_1': 0.865, 'page_3_5': 0.455, 'striking': 0.0, 'top_3': 0.0}
  page_1   : median=0.865  zero_share=0.486  share landing in the open window (0, median)=0.0140
  page_3_5 : median=0.455  zero_share=0.438  share landing in the open window (0, median)=0.0625
  striking : median=0.000  zero_share=0.583  share landing in the open window (0, median)=0.0000
  top_3    : median=0.000  zero_share=0.590  share landin

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Trust the queue's ranking (score = gap × volume), but not the story I told about *why* volume dominates it — big pages in this dataset skew toward better CTR, not worse, so volume-weighting is an impact filter, not a quality signal. And the `weak_engagement_confirms` reason code needs the fix above before a content team should treat it as a second opinion: as originally coded it was silent on two of three position tiers and barely spoke on the third, which is worse than not having it at all, since a reason code that looks like corroboration but rarely fires reads as confirmation by omission. I'll carry the conditional-median fix back into `w04_baseline_score.ipynb`.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.